<div style="
    width:100%;
    max-width:1080px;
    margin:24px 0 32px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #c2410c 0%, #f59e0b 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h1 style="margin:0 0 14px 0; font-size:34px; line-height:1.2; font-weight:700; color:#ffffff;">La Preparación de los Datasets</h1>
<p style="margin:0; color:#cbd5e1; font-size:18px;">Ningún fine-tuning funciona bien si los datos no están preparados en el formato adecuado para el modelo y el entrenador. Este notebook se centra en esa adaptación previa del dataset.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Por qué importa</div>
    <div style="color:#f8fafc;">El rendimiento del modelo depende tanto de la arquitectura como de la calidad y coherencia del flujo de datos.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Qué vas a aprender aquí</div>
    <div style="color:#f8fafc;">A entender cómo preparar datasets para que encajen con tokenización, entrenamiento y evaluación.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Regla práctica que debes recordar</div>
    <div style="color:#f8fafc;">Un pipeline de entrenamiento sólido empieza por un dataset bien estructurado, no por hiperparámetros sofisticados.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Contexto mínimo</div>
    <div style="color:#f8fafc;">Preparar datasets implica limpiar, estructurar, tokenizar y dejar listas las particiones para el entrenamiento.</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:36px 0 42px 0;">

In [ ]:
# Bloque 1 — Definición de los datos crudos
"""Este bloque define los datos crudos que serán preparados como dataset.
Definimos un dataset mínimo con textos y etiquetas para el fine-tuning.
La variable "textos" contiene una lista con los textos de entrada para el fine-tuning.
La variable "etiquetas" contiene una lista de etiquetas correspondientes a cada uno de los textos de entrada.
Estas estructuras representan el punto inicial del proceso de preparación del dataset."""

textos = ["El producto es excelente.", "La atención fue muy mala.", "El envío llegó rápido.", "La calidad es deficiente."]
etiquetas = [1, 0, 1, 0]

textos, etiquetas # Muestra los textos y etiquetas definidos para verificar su contenido.


In [ ]:
# Bloque 2 — Carga del tokenizer
"""Este bloque carga un tokenizer preentrenado desde Hugging Face.
Este bloque importa "AutoTokenizer" de "transformers" para cargar un tokenizador preentrenado de Hugging Face.

Aplicamos la clase "AutoTokenizer", acompañada del método ".from_pretrained()", a la variable "tokenizador" para cargar un tokenizador preentrenado.
El argumento "bert-base-uncased" especifica el modelo preentrenado que se desea utilizar.
La variable "tokenizador" almacena una instancia del tokenizador preentrenado lista para ser utilizada en tareas de clasificación de secuencias.
El tokenizador está entrenado para inglés, mientras que los textos del ejemplo están en español."""

from transformers import AutoTokenizer

tokenizador = AutoTokenizer.from_pretrained("bert-base-uncased")

tokenizador # Muestra el tokenizador cargado para verificar su contenido.


In [ ]:
# Bloque 3 — Construcción del dataset inicial
"""Este bloque construye un dataset a partir de los textos y etiquetas.
Este bloque importa "Dataset" de "datasets" para construir un dataset compatible con Hugging Face.

Definimos el diccionario que contiene los textos y etiquetas a partir del cual se construye el dataset.
La variable "diccionario" almacena los textos y etiquetas que se utilizarán para construir el dataset.

Aplicamos la clase "Dataset", acompañada del método ".from_dict()", a la variable "dataset" para crear el dataset.
El argumento "diccionario" es el diccionario definido previamente a partir del cual se construye el dataset.
La variable "dataset" contiene el dataset sin tokenizar, listo para ser procesado con el método ".map()"."""

from datasets import Dataset

diccionario = {"texto": textos, "label": etiquetas}

dataset = Dataset.from_dict(diccionario)

dataset # Muestra el dataset inicial sin tokenizar para verificar su contenido.


In [ ]:
# Bloque 4 — Tokenización del dataset
"""Este bloque aplica tokenización al dataset.
Definimos la función "def tokenize_batch(batch)" para tokenizar cada lote de textos del dataset.
El parámetro "batch" es un diccionario que contiene los textos asociados a la clave "texto" del diccionario definido previamente.

Esta función retorna la llamada al tokenizador "tokenizador()" con los textos del lote, aplicando padding y truncado.
El argumento "batch["texto"]" contiene los textos del lote asociados a la clave "texto".
El parámetro "padding=" rellena las secuencias hasta la longitud de la más larga del lote.
El parámetro "truncation=" trunca las secuencias que exceden la longitud máxima del modelo.

Aplicamos el método ".map()" al dataset para tokenizar sus textos.
El argumento "tokenize_batch" es la función que se aplica a cada lote para realizar la tokenización.
El parámetro "batched=" indica que la función se aplicará a lotes de ejemplos en lugar de a ejemplos individuales.
La variable "dataset_tokenizado" contiene ahora el dataset tokenizado, listo para ser formateado con el método ".set_format()"."""

def tokenize_batch(batch):
    return tokenizador(batch["texto"], padding=True, truncation=True)

dataset_tokenizado = dataset.map(tokenize_batch, batched=True)

dataset_tokenizado # Muestra una representación del dataset tokenizado. 


In [ ]:
# Bloque 5 — Formateo del dataset para PyTorch
"""Este bloque convierte el dataset tokenizado en formato compatible con PyTorch.
Definimos una lista de columnas que serán utilizadas para convertir el dataset en tensores.
La variable "lista_columnas" contiene la lista de columnas que serán convertidas en tensores.

Aplicamos el método ".set_format()" al dataset tokenizado para establecer el formato de salida.
El parámetro "type=" indica con el valor "torch" que los datos se devolverán como tensores de PyTorch.
El parámetro "columns=" especifica las columnas que se incluirán en la salida, en este caso, las columnas definidas en la lista "lista_columnas".
La variable "dataset_tokenizado" almacena ahora el dataset tokenizado en un formato compatible con PyTorch.
Esto permite utilizar el dataset directamente en modelos de PyTorch para entrenamiento o inferencia."""

lista_columnas = ["input_ids", "attention_mask", "label"]

dataset_tokenizado.set_format(type="torch", columns=lista_columnas)

dataset_tokenizado # Muestra el dataset tokenizado en formato compatible con PyTorch.


In [ ]:
# Bloque 6 — División del dataset
"""Este bloque divide el dataset en particiones de entrenamiento y validación.
Aplicamos el método ".train_test_split()" al dataset tokenizado para dividirlo en conjuntos de entrenamiento y validación.
El parámetro "test_size=" indica con el valor 0.25 que el 25 % de los datos se utilizarán para validación y el 75 % restante para entrenamiento.
La variable "particiones" almacena ahora las divisiones resultantes en un diccionario con las claves "train" y "test".
Esta partición permite entrenar el modelo con el conjunto de entrenamiento y evaluar su rendimiento con el conjunto de validación.
La división es aleatoria y no fija una semilla ni estratifica las etiquetas; con solo cuatro ejemplos, la validación contiene un único ejemplo."""

particiones = dataset_tokenizado.train_test_split(test_size=0.25)

particiones # Muestra las particiones del dataset: entrenamiento y validación.


In [ ]:
# Bloque 7 — Resultados finales
"""Este bloque organiza los resultados del proceso de preparación del dataset.
La variable "resultado" almacena en forma de diccionario el dataset tokenizado y las particiones generadas."""

resultado = {
    "dataset_tokenizado": dataset_tokenizado,
    "train_dataset": particiones["train"],     # Extraemos el valor correspondiente a la partición de entrenamiento.
    "validation_dataset": particiones["test"]  # Extraemos el valor correspondiente a la partición de validación.
}

resultado # Muestra el diccionario con el dataset tokenizado y las particiones de entrenamiento y validación.


<div style="
    width:100%;
    max-width:1080px;
    margin:40px 0 24px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #0f766e 0%, #2dd4bf 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h2 style="margin:0 0 14px 0; font-size:30px; line-height:1.2; font-weight:700; color:#ffffff;">Conclusiones</h2>
<p style="margin:0; color:#cbd5e1; font-size:18px;">En este notebook has visto que preparar correctamente los datasets es parte central del ajuste fino y no un paso secundario.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Qué conviene recordar</div>
    <div style="color:#f8fafc;">La calidad del entrenamiento depende en gran medida de cómo llegan los ejemplos al modelo.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Error frecuente a evitar</div>
    <div style="color:#f8fafc;">Confiar en el modelo y el framework mientras se descuida la coherencia de los datos de entrada.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente paso natural</div>
    <div style="color:#f8fafc;">Detenerse en la Configuración del Entrenamiento para dar continuidad directa al hilo del temario.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Conexión con el siguiente concepto</div>
    <div style="color:#f8fafc;">El siguiente notebook se centra en la Configuración del Entrenamiento y amplía de forma natural lo que acabas de trabajar.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente notebook del temario</div>
    <div style="color:#f8fafc;">06_Fine_tuning_con_trainer/03_La_configuracion_del_entrenamiento.ipynb</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:42px 0 24px 0;">